# 从补丁建议到可验收改动：Code Agent 与执行反馈

候选补丁通过了它自己编写的测试，为什么仍不能交付？今天读文件、执行测试，
再比较一次补丁、有界修复与独立验收拒绝。默认离线，使用预先写好的候选，
不测量模型能力提升、不更新参数。学生只实现选择条件、恢复判断与有限循环；
路径、测试进程和文件写入由公开脚手架提供。

四个跨媒介路线锚点是：work-unit-contract → policy-before-side-effect →
postcondition-gate → recovery-needs-fingerprint；repair 与 mini-project 是其中的练习阶段。
最后的 TODO 完成前会停止；前面实验可以顺序运行。仅操作提供的合成临时仓库。

In [ ]:
import json
import shutil
import tempfile
from pathlib import Path
from contextlib import ExitStack
from projects.capstones.code_agent.public_api import (
    CodeTask, PatchOperation, PatchPlan, SafeWorkspace,
    WorkspaceViolation, PatchFailure, TestStatus,
)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "projects/capstones/code_agent/fixtures").exists())
FIXTURE = ROOT / "projects/capstones/code_agent/fixtures/calculator_repo"
TEMP = ROOT / "tmp/lesson04-notebook"
TEMP.mkdir(parents=True, exist_ok=True)
sessions = ExitStack()

def fresh():
    directory = sessions.enter_context(tempfile.TemporaryDirectory(dir=TEMP))
    root = Path(directory) / "repo"
    shutil.copytree(FIXTURE, root)
    return SafeWorkspace(root, allowed_paths=["src", "tests"])

task = CodeTask(task_id="lesson04-add", issue="Return the sum for all integer inputs.",
                test_command=["python", "-m", "pytest", "-q", "tests"],
                allowed_paths=["src", "tests"], requires_approval=False)
print({"fixture": task.fixture_version, "network_required": False})

## 先测未修改的仓库

预测 `add(2, 3)` 与 `add(9, 0)` 的返回值。基线包含一个失败行为和一个必须保留的
正确行为。观察 stdout 的失败位置和进程返回码，再比较模型所说的“完成”。
返回 Slide 的 work-unit-contract，说明 before、after、diff、test 各证明什么。

In [ ]:
workspace = fresh()
before = workspace.snapshot(fixture_version=task.fixture_version)
print(workspace.read_text("src/calculator.py"))
baseline = workspace.run_tests(task.test_command)
print({"status": baseline.status.value, "return_code": baseline.return_code})
print(baseline.stdout[-900:])
assert baseline.status is TestStatus.FAILED
assert "1 failed, 1 passed" in baseline.stdout

## 观察是环境的切片

`list_files/read_text/search` 给出可见文本；`snapshot` 给出文件指纹。
指纹只回答文件是否变化，不回答计算是否正确。接口实现位于
`projects/capstones/code_agent/workspace.py`，对象位于 `contracts.py`。
前课只复用“有限步骤与 stop reason”的概念，无需运行前课 Notebook。

In [ ]:
view = workspace.view()
print({"files": view.list_files(), "matches": view.search("return")})
assert not hasattr(view, "apply_plan")
assert workspace.snapshot().digest == before.digest

## 策略检查早于写入

预测：tests 可读是否等于可改？本课开发测试可读但不可写；敏感 protected_paths
读写都拒绝。命令不能加载插件、替换配置或缩小测试选择。Python facade 只限制
工具接口，不构成 OS 沙箱；不要用它运行未知仓库。

In [ ]:
blocked = []
for path, old, new in [("../outside.py", "x", "y"),
                       ("tests/test_calculator.py", "== 5", "!= 5")]:
    try:
        workspace.apply(PatchOperation(path=path, old=old, new=new))
    except WorkspaceViolation:
        blocked.append(path)
try:
    workspace.run_tests(["python", "-m", "pytest", "-p", "custom_plugin"])
except WorkspaceViolation:
    blocked.append("test_plugin")
assert len(blocked) == 3 and workspace.snapshot().digest == before.digest
print({"blocked": blocked, "write_occurred": False})

## 整份补丁检查完才能写入

精确替换要求旧文本恰好出现指定次数。零次通常表示基线过期，多次可能定位含糊。
下例第一个文件合法，第二个目标是越权测试。预测第一个修改会不会残留。
本工具只支持现有 UTF-8 文件替换，不支持新增、删除或二进制文件。

In [ ]:
try:
    workspace.apply_plan([
        PatchOperation(path="src/calculator.py", old="return a - b", new="return a * b"),
        PatchOperation(path="tests/test_calculator.py", old="== 5", new="!= 5"),
    ])
except WorkspaceViolation as error:
    print({"status": "rejected", "reason": str(error)})
assert not workspace.diff() and workspace.snapshot().digest == before.digest

## 失败反馈决定下一次修复

直接读下面的有限循环：先试乘法，读取真实失败，恢复原文件，再试加法。
同一基线、同一开发测试、最多两次尝试。最后的绿灯仍只表示开发检查通过。
第二个候选是预先写好的教学材料，不是模型从反馈自主学到的新能力。

In [ ]:
previous = baseline
trajectory = []
candidates = ["return a * b", "return a + b"]
for attempt, replacement in enumerate(candidates[:2], 1):
    assert previous.status is TestStatus.FAILED
    assert workspace.snapshot().digest == before.digest
    workspace.apply(
        PatchOperation(path="src/calculator.py", old="return a - b", new=replacement)
    )
    result = workspace.run_tests(task.test_command)
    trajectory.append({"attempt": attempt, "feedback": previous.status.value,
                       "test": result.status.value, "diff": workspace.diff()})
    if result.status is TestStatus.PASSED:
        break
    workspace.rollback()
    previous = result
print(json.dumps(trajectory, indent=2))
assert len(trajectory) == 2 and result.status is TestStatus.PASSED
workspace.rollback()

## 测试绿灯与独立验收分开

下面的补丁只迎合已知输入。先预测开发测试是否能识别，再解释任意整数相加的
业务要求。完整项目把保留验收放在候选目录外，在开发检查通过后运行一次；
失败结束本轮，不把具体输出交回 planner。反复看它调代码会污染保留集。

In [ ]:
overfit = fresh()
overfit.apply(PatchOperation(path="src/calculator.py", old="return a - b",
    new="return {(2, 3): 5, (-2, 1): -1}.get((a, b), a)"))
result = overfit.run_tests(task.test_command)
assert result.status is TestStatus.PASSED
print({"development": result.status.value, "acceptance": "not_checked",
       "decision": "candidate_only"})
overfit.rollback()

## 原有未提交改动属于恢复点

脏工作区指已有未提交改动。基线应是任务进入时的字节内容，不能默认恢复 Git HEAD。
下例用户笔记与 CRLF 换行都必须保留；回滚失败候选不等于清空整个目录。

In [ ]:
dirty = fresh()
source = dirty.root / "src/calculator.py"
original = b"# User note\r\ndef add(a, b):\r\n    return a - b\r\n"
source.write_bytes(original)
dirty_before = dirty.snapshot().digest
dirty.apply(PatchOperation(path="src/calculator.py", old="return a - b", new="return a * b"))
assert dirty.run_tests(task.test_command).status is TestStatus.FAILED
dirty.rollback()
assert source.read_bytes() == original and dirty.snapshot().digest == dirty_before
print({"user_note_preserved": True, "line_endings_preserved": True})

## 再次观察后才能恢复

若候选产生后用户又改了同一文件，不得直接覆盖。checkpoint 还需要任务策略指纹、
fixture、累计资源与下一步。本课 journal 在内存中，活跃候选缺失 journal 时拒绝
跨进程恢复，不能只凭 digest 猜回原字节。观察下面保留现场的拒绝。

In [ ]:
external = fresh()
external.apply(PatchOperation(path="src/calculator.py", old="return a - b", new="return a + b"))
user_edit = "# Another user edit\n" + external.read_text("src/calculator.py")
(external.root / "src/calculator.py").write_bytes(user_edit.encode())
try:
    external.rollback()
except WorkspaceViolation as error:
    print({"stop_reason": "unsafe_state", "reason": str(error)})
assert external.read_text("src/calculator.py") == user_edit

## 你拥有选择条件与停止决定

实现 select_candidate：开发测试、独立验收、范围三项明确通过才能选择。
未检查必须拒绝。随后到 starter/agent.py 完成有限循环和恢复判断；无需重写
路径解析器、子进程管理或模型 SDK。测试不能随候选更改。预计课后 1.5 小时。

In [ ]:
def select_candidate(development: str, acceptance: str, allowed: bool) -> bool:
    # TODO: require all three independent conditions.
    raise NotImplementedError("implement candidate selection")

In [ ]:
assert select_candidate("passed", "passed", True)
assert not select_candidate("passed", "not_checked", True)
assert not select_candidate("failed", "passed", True)
assert not select_candidate("passed", "passed", False)
print({"selection_contract": "passed"})
sessions.close()

## 回到开场，再迁移到数字界面

验收文件被修改，先在写入策略拒绝；基线过期，重新读取并停止旧补丁；无关改动
丢失，恢复与完整性检查失败。API 200、DOM 成功提示、截图对勾也需后端状态检查。
课程进程不等于 OS 沙箱；真实系统另需资源、网络、凭据、依赖与并发隔离。

[CS329A](https://cs329a.stanford.edu/) 的验证/反馈路线、
[CS329Z](https://cs329z.stanford.edu/) 的组件优先课程表、
[SWE-agent](https://arxiv.org/abs/2405.15793) 与
[CodeMonkeys](https://arxiv.org/abs/2501.14723) 提供延伸阅读。
本课不比较模型能力或实际开发省时。更多来源见 student/resource-pack.md。